# ChakraModel — Full Universal Evaluation v8
## All 15 Datasets · Image + Video · YOLO Fallback Fixed · Full Comparison Table

**What's new in v8:**
- ✅ YOLO fallback: 0 detections → full-image ROI (fixes all the 0.0000 FAILs from v7)
- ✅ Video evaluation: PolypGen, LDPolyp, HyperKvasir videos, CVC video
- ✅ LDPolyp-without-polyps → specificity check (false positive rate)
- ✅ LDPolyp-with-polyps → sensitivity check (dice/IoU per frame)
- ✅ Auto-detects all 15 attached datasets by type
- ✅ Full comparison table with per-dataset dice ± std, IoU, detection rate
- ✅ Saves combined_report_v8.json + comparison_table.csv


In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
    "ultralytics", "thop", "numpy", "opencv-python",
    "matplotlib", "pandas", "tabulate", "--quiet"], check=False)
print("Packages ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 5.0 MB/s eta 0:00:00
Packages ready.


In [2]:
import os, sys, glob, cv2, json, time, shutil, traceback
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

WORKING_DIR = "/kaggle/working"
INPUT_DIR   = "/kaggle/input"
print(f"Working dir : {WORKING_DIR}")
print(f"Input dir   : {INPUT_DIR}")
print(f"Datasets attached: {len(os.listdir(INPUT_DIR))}")
for d in sorted(os.listdir(INPUT_DIR)):
    print(f"  /kaggle/input/{d}")

Working dir : /kaggle/working
Input dir   : /kaggle/input
Datasets attached: 1
  /kaggle/input/datasets


In [3]:
# ─────────────────────────────────────────────────────────────
# Dataset map — 15 datasets classified by type
# Update DATASET_MAP keys if Kaggle renames the mount points.
# ─────────────────────────────────────────────────────────────

DATASET_MAP = {
    # ── CODE & WEIGHTS ──────────────────────────────────────────
    "chakramodel-kaggle-code":       {"role": "code"},
    "finalmuruga-harae":             {"role": "weights"},
    "chakratransformer-weights":     {"role": "weights"},
    "final-om-evlautation-upload":   {"role": "eval_output"},
    "om-finalkaggle-upload":         {"role": "eval_output"},

    # ── IMAGE datasets (with GT masks) ──────────────────────────
    "endoscene-cvc300-polyp-raw-dataset": {
        "role": "image", "name": "CVC-300",
        "img_sub": "CVC-300/Original",
        "mask_sub": "CVC-300/Ground Truth",
    },
    "chakramodel-evaluation-datasets": {
        "role": "image_multi", "name": "Eval Pack",
        "sub_datasets": [
            {"name": "CVC-ClinicDB",  "img_sub": "cvc-clinicdb/images", "mask_sub": "cvc-clinicdb/masks"},
            {"name": "Kvasir-SEG",    "img_sub": "kvasir-seg/images",   "mask_sub": "kvasir-seg/masks"},
            {"name": "ETIS-LARIB",    "img_sub": "etis-larib/images",   "mask_sub": "etis-larib/masks"},
        ],
    },
    "hyperkvasir-dataset-first-half-and-and-ld-dataset": {
        "role": "image_multi", "name": "HyperKvasir+LD Images",
        "sub_datasets": [
            {"name": "HyperKvasir-Seg",
             "img_sub": "hyper-kvasir-segmented-images-part 3/segmented-images/images",
             "mask_sub": "hyper-kvasir-segmented-images-part 3/segmented-images/masks"},
        ],
    },
    "polypdb-polyp-raw": {
        "role": "image_multi", "name": "PolypDB",
        "sub_datasets": [
            {"name": "PolypDB/Simula/NBI",   "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/Simula/NBI/images",   "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/Simula/NBI/masks"},
            {"name": "PolypDB/Simula/WLI",   "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/Simula/WLI/images",   "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/Simula/WLI/masks"},
            {"name": "PolypDB/BKAI/WLI",     "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/WLI/images",     "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/WLI/masks"},
            {"name": "PolypDB/BKAI/NBI",     "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/NBI/images",     "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/NBI/masks"},
            {"name": "PolypDB/BKAI/BLI",     "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/BLI/images",     "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/BLI/masks"},
            {"name": "PolypDB/BKAI/FICE",    "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/FICE/images",    "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/FICE/masks"},
            {"name": "PolypDB/BKAI/LCI",     "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/LCI/images",     "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/LCI/masks"},
            {"name": "PolypDB/Karolinska",   "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/Karolinska/WLI/images","mask_sub":"PolypDB/PolypDB/PolypDB_center_wise/Karolinska/WLI/masks"},
            {"name": "PolypDB/FICE",         "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/FICE/images",       "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/FICE/masks"},
            {"name": "PolypDB/BLI",          "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/BLI/images",        "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/BLI/masks"},
            {"name": "PolypDB/NBI",          "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/NBI/images",        "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/NBI/masks"},
            {"name": "PolypDB/WLI",          "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/WLI/images",        "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/WLI/masks"},
            {"name": "PolypDB/LCI",          "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/LCI/images",        "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/LCI/masks"},
        ],
    },

    # ── VIDEO datasets ──────────────────────────────────────────
    "polypgen20021-video": {
        "role": "video_positive", "name": "PolypGen2021-Video",
        "note": "Polyps present — dice/IoU per frame",
    },
    "ldpolypvideowithoutpolyps": {
        "role": "video_negative", "name": "LDPolyp-NoPolyp",
        "note": "NO polyps — specificity check (false positive rate)",
    },
    "ldpolypvideopolyponly": {
        "role": "video_positive", "name": "LDPolyp-PolypOnly",
        "note": "Polyps only — sensitivity check (dice/IoU per frame)",
    },
    "hperkvasir-labeled-videos-part2-002": {
        "role": "video_positive", "name": "HyperKvasir-Video-P2B",
        "note": "HyperKvasir labeled video frames part 2 (B)",
    },
    "hyperkvasir-labeled-videos-part2-001": {
        "role": "video_positive", "name": "HyperKvasir-Video-P2A",
        "note": "HyperKvasir labeled video frames part 2 (A)",
    },
    "cvc-sample-video": {
        "role": "video_positive", "name": "CVC-Video",
        "note": "CVC sample video clips",
    },
}

# Resolve actual paths
def resolve_dataset(ds_key, ds_cfg):
    for name in os.listdir(INPUT_DIR):
        if ds_key in name or name in ds_key:
            return os.path.join(INPUT_DIR, name)
    # Try partial match
    for name in os.listdir(INPUT_DIR):
        if any(part in name for part in ds_key.split("-") if len(part) > 4):
            return os.path.join(INPUT_DIR, name)
    return None

RESOLVED = {}
print("\nDataset resolution:")
for key, cfg in DATASET_MAP.items():
    path = resolve_dataset(key, cfg)
    RESOLVED[key] = path
    role = cfg["role"]
    status = "✓ found" if path else "✗ not found"
    print(f"  [{role:15s}] {key:50s} {status}")



Dataset resolution:
  [code           ] chakramodel-kaggle-code                            ✗ not found
  [weights        ] finalmuruga-harae                                  ✗ not found
  [weights        ] chakratransformer-weights                          ✗ not found
  [eval_output    ] final-om-evlautation-upload                        ✗ not found
  [eval_output    ] om-finalkaggle-upload                              ✗ not found
  [image          ] endoscene-cvc300-polyp-raw-dataset                 ✓ found
  [image_multi    ] chakramodel-evaluation-datasets                    ✓ found
  [image_multi    ] hyperkvasir-dataset-first-half-and-and-ld-dataset  ✓ found
  [image_multi    ] polypdb-polyp-raw                                  ✗ not found
  [video_positive ] polypgen20021-video                                ✗ not found
  [video_negative ] ldpolypvideowithoutpolyps                          ✗ not found
  [video_positive ] ldpolypvideopolyponly                              ✗ not f

In [4]:
# ─────────────────────────────────────────────────────────────
# Load YOLO + ChakraNet Transformer
# ─────────────────────────────────────────────────────────────

# 1. Find & copy source code
CODE_PATH = None
for root, dirs, files in os.walk(INPUT_DIR):
    if "chakranet_segmenter.py" in files and os.path.basename(root) == "src":
        CODE_PATH = os.path.dirname(root)
        break

if CODE_PATH:
    shutil.copytree(CODE_PATH, f"{WORKING_DIR}/chakramodel", dirs_exist_ok=True)
    os.chdir(f"{WORKING_DIR}/chakramodel")
    if f"{WORKING_DIR}/chakramodel/src" not in sys.path:
        sys.path.insert(0, f"{WORKING_DIR}/chakramodel/src")
    print(f"Source code loaded from: {CODE_PATH}")
else:
    print("WARNING: chakranet_segmenter.py not found — segmenter will be mocked.")

# 2. Find weights
os.makedirs(f"{WORKING_DIR}/chakramodel/weights", exist_ok=True)

TRANSFORMER_PTH = None
YOLO_PT         = None

for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if f == "chakra_transformer_best.pth" and TRANSFORMER_PTH is None:
            TRANSFORMER_PTH = os.path.join(root, f)
        if f == "best.pt" and YOLO_PT is None:
            YOLO_PT = os.path.join(root, f)

if TRANSFORMER_PTH:
    shutil.copy(TRANSFORMER_PTH, f"{WORKING_DIR}/chakramodel/weights/chakra_transformer_best.pth")
    print(f"Transformer weights: {TRANSFORMER_PTH}")
else:
    print("WARNING: chakra_transformer_best.pth not found.")

if YOLO_PT:
    shutil.copy(YOLO_PT, f"{WORKING_DIR}/chakramodel/weights/best.pt")
    YOLO_PT_LOCAL = f"{WORKING_DIR}/chakramodel/weights/best.pt"
    print(f"YOLO weights: {YOLO_PT}")
else:
    print("WARNING: best.pt not found.")
    YOLO_PT_LOCAL = None

# 3. Load YOLO
YOLO_MODEL = None
if YOLO_PT_LOCAL:
    try:
        from ultralytics import YOLO
        YOLO_MODEL = YOLO(YOLO_PT_LOCAL)
        print("YOLO loaded ✓")
    except Exception as e:
        print(f"YOLO load failed: {e}")

# 4. Load ChakraNet segmenter
SEG_MODEL = None
try:
    from chakranet_segmenter import ChakraNetSegmenter
    SEG_MODEL = ChakraNetSegmenter(
        weights_path=f"{WORKING_DIR}/chakramodel/weights/chakra_transformer_best.pth"
    )
    print("ChakraNetSegmenter loaded ✓")
except Exception as e:
    print(f"Segmenter load failed: {e}")
    print("Will use fallback mock segmenter for testing.")


Source code loaded from: /kaggle/input/datasets/gokulrocky/om-finalkaggle-upload
Transformer weights: /kaggle/input/datasets/gokulraj324/chakramodel-weightsupdated4/chakra_transformer_best.pth
YOLO weights: /kaggle/input/datasets/gokulraj324/chakramodel-weightsupdated4/best.pt
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
YOLO loaded ✓
2026-09-08 08:32:33,882 [HW-MONITOR] INFO WARMUP phase — GPU capped at 40% (~1.6 GB) | GPU ramp-up starts at 120s | Full boost at 240s
Segmenter load failed: cannot import name 'ChakraNetSegmenter' from 'chakranet_segmenter' (/kaggle/working/chakramodel/src/chakranet_segmenter.py)
Will use fallback mock segmenter for testing.


In [5]:
# ─────────────────────────────────────────────────────────────
# Core evaluation utilities
# ─────────────────────────────────────────────────────────────

IMG_EXTS  = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
VID_EXTS  = {".mp4", ".avi", ".mov", ".mkv"}

def dice_iou(pred_bin, gt_bin):
    pred = pred_bin.astype(bool)
    gt   = gt_bin.astype(bool)
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    dice  = 2 * inter / (pred.sum() + gt.sum() + 1e-8)
    iou   = inter / (union + 1e-8)
    return float(dice), float(iou)


def run_yolo_segment(img_bgr, yolo_model, seg_model, fallback=True):
    """
    Run YOLO detect → crop → segment_roi().
    If YOLO finds nothing and fallback=True, uses full image as ROI.
    Returns (pred_mask_hw, n_detections, mode).
    """
    h, w = img_bgr.shape[:2]
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    mode      = "none"

    # YOLO
    boxes = []
    if yolo_model is not None:
        try:
            res = yolo_model(img_bgr, verbose=False)
            if len(res) > 0 and res[0].boxes is not None and len(res[0].boxes) > 0:
                boxes = res[0].boxes.xyxy.cpu().numpy().tolist()
        except Exception:
            pass

    if len(boxes) == 0:
        if fallback:
            boxes = [[0, 0, w, h]]
            mode  = "fallback"
        else:
            return pred_mask, 0, "no_detection"
    else:
        mode = "yolo"

    if seg_model is None:
        # Mock: return empty mask (for testing pipeline without weights)
        return pred_mask, len(boxes), mode + "_mock"

    for box in boxes:
        x1, y1, x2, y2 = [int(v) for v in box]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        crop = img_bgr[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        try:
            result = seg_model.segment_roi(crop)
            if result is None:
                continue
            # segment_roi returns dict with "mask" key (or numpy array)
            if isinstance(result, dict):
                crop_mask = result.get("mask", result.get("seg_mask", None))
            elif isinstance(result, np.ndarray):
                crop_mask = result
            else:
                continue
            if crop_mask is None:
                continue
            crop_h, crop_w = y2 - y1, x2 - x1
            if crop_mask.shape != (crop_h, crop_w):
                crop_mask = cv2.resize(
                    crop_mask.astype(np.uint8), (crop_w, crop_h),
                    interpolation=cv2.INTER_NEAREST
                )
            pred_mask[y1:y2, x1:x2] = np.maximum(
                pred_mask[y1:y2, x1:x2], crop_mask.astype(np.uint8)
            )
        except Exception as ex:
            pass

    return pred_mask, len(boxes), mode


def find_mask_for_image(img_path, mask_root):
    """Find the GT mask file for a given image path."""
    stem = Path(img_path).stem
    for ext in [".png", ".jpg", ".bmp", ".tif"]:
        candidate = os.path.join(mask_root, stem + ext)
        if os.path.exists(candidate):
            return candidate
    # Search recursively
    for root, _, files in os.walk(mask_root):
        for f in files:
            if Path(f).stem == stem:
                return os.path.join(root, f)
    return None


def auto_find_img_mask_dirs(dataset_root):
    """
    Auto-detect images/ and masks/ subdirectories in a dataset root.
    Returns list of (img_dir, mask_dir, label) tuples.
    """
    found = []
    for root, dirs, files in os.walk(dataset_root):
        img_exts_here = sum(1 for f in files if Path(f).suffix.lower() in IMG_EXTS)
        if img_exts_here < 3:
            continue
        # Look for sibling mask directory
        parent = os.path.dirname(root)
        base   = os.path.basename(root).lower()
        if any(k in base for k in ("image", "img", "frame", "original", "photo")):
            mask_candidates = ["masks", "mask", "gt", "ground_truth",
                               "groundtruth", "annotation", "ann", "label"]
            for mc in mask_candidates:
                mp = os.path.join(parent, mc)
                if os.path.isdir(mp):
                    found.append((root, mp, os.path.basename(parent)))
                    break
    return found if found else []


print("Evaluation utilities loaded ✓")


Evaluation utilities loaded ✓


In [6]:
# ─────────────────────────────────────────────────────────────
# Image dataset evaluator
# ─────────────────────────────────────────────────────────────

def evaluate_image_dataset(name, img_dir, mask_dir, max_images=500):
    """
    Evaluate ChakraModel on an image segmentation dataset.
    Returns a result dict.
    """
    if not os.path.isdir(img_dir):
        return {"name": name, "status": "skip", "reason": f"img_dir not found: {img_dir}"}
    if not os.path.isdir(mask_dir):
        return {"name": name, "status": "skip", "reason": f"mask_dir not found: {mask_dir}"}

    img_files = sorted([
        f for f in os.listdir(img_dir)
        if Path(f).suffix.lower() in IMG_EXTS
    ])[:max_images]

    if not img_files:
        return {"name": name, "status": "skip", "reason": "no image files found"}

    dices, ious = [], []
    n_yolo, n_fallback, n_skip = 0, 0, 0
    per_image = []

    for fname in img_files:
        img_path  = os.path.join(img_dir, fname)
        mask_path = find_mask_for_image(img_path, mask_dir)
        if mask_path is None:
            n_skip += 1
            continue

        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            n_skip += 1
            continue
        gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if gt is None:
            n_skip += 1
            continue

        pred_mask, n_boxes, mode = run_yolo_segment(
            img_bgr, YOLO_MODEL, SEG_MODEL, fallback=True
        )

        gt_bin   = (gt > 127).astype(np.uint8)
        pred_bin = (pred_mask > 127).astype(np.uint8)
        d, iou   = dice_iou(pred_bin, gt_bin)
        dices.append(d)
        ious.append(iou)
        if "fallback" in mode: n_fallback += 1
        elif "yolo" in mode:   n_yolo += 1
        per_image.append({"file": fname, "dice": round(d,4), "iou": round(iou,4), "mode": mode})

    if not dices:
        return {"name": name, "status": "no_valid_pairs"}

    return {
        "name":         name,
        "type":         "image",
        "status":       "done",
        "n_total":      len(img_files),
        "n_evaluated":  len(dices),
        "n_skip":       n_skip,
        "dice_mean":    round(float(np.mean(dices)), 4),
        "dice_std":     round(float(np.std(dices)),  4),
        "dice_median":  round(float(np.median(dices)),4),
        "iou_mean":     round(float(np.mean(ious)),  4),
        "iou_std":      round(float(np.std(ious)),   4),
        "yolo_detections": n_yolo,
        "fallback_used":   n_fallback,
        "per_image":    per_image,
    }

print("Image evaluator ready ✓")


Image evaluator ready ✓


In [7]:
# ─────────────────────────────────────────────────────────────
# Video dataset evaluator
# ─────────────────────────────────────────────────────────────

def evaluate_video_dataset(name, dataset_root, is_negative=False,
                            max_videos=20, max_frames_per_video=150, sample_every=5):
    """
    Evaluate ChakraModel on a video dataset.
    - is_negative=True → no GT masks, compute false-positive rate
    - is_negative=False → look for GT masks per frame, compute dice/IoU
    """
    if not os.path.isdir(dataset_root):
        return {"name": name, "status": "skip", "reason": f"root not found: {dataset_root}"}

    # Find video files
    video_files = []
    for ext in VID_EXTS:
        video_files += glob.glob(os.path.join(dataset_root, f"**/*{ext}"), recursive=True)
    video_files = sorted(video_files)[:max_videos]

    # Also find frame image directories (some video datasets are pre-extracted)
    frame_dirs = auto_find_img_mask_dirs(dataset_root)

    dices, ious = [], []
    fp_frames, total_frames = 0, 0
    n_videos_processed = 0

    # ── Pre-extracted frames mode ──────────────────────────────
    if frame_dirs and not is_negative:
        for img_dir, mask_dir, label in frame_dirs:
            res = evaluate_image_dataset(
                f"{name}/{label}", img_dir, mask_dir,
                max_images=max_frames_per_video
            )
            if res.get("status") == "done":
                dices  += [p["dice"] for p in res["per_image"]]
                ious   += [p["iou"]  for p in res["per_image"]]
                n_videos_processed += 1
        if dices:
            return {
                "name": name, "type": "video_frames",
                "status": "done", "mode": "pre_extracted_frames",
                "n_frame_sets":   n_videos_processed,
                "n_frames_total": len(dices),
                "dice_mean":  round(float(np.mean(dices)), 4),
                "dice_std":   round(float(np.std(dices)),  4),
                "dice_median":round(float(np.median(dices)),4),
                "iou_mean":   round(float(np.mean(ious)),  4),
                "iou_std":    round(float(np.std(ious)),   4),
                "is_negative": False,
            }

    # ── Raw video mode ─────────────────────────────────────────
    if not video_files:
        # Last resort: treat root as flat image directory
        img_files = [f for f in os.listdir(dataset_root)
                     if Path(f).suffix.lower() in IMG_EXTS]
        if img_files:
            return evaluate_image_dataset(
                name, dataset_root, dataset_root,  # no separate mask dir
                max_images=max_frames_per_video
            )
        return {"name": name, "status": "skip", "reason": "no video or image files found"}

    for vid_path in video_files:
        cap = cv2.VideoCapture(vid_path)
        if not cap.isOpened():
            continue
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fid   = 0
        while fid < min(total, max_frames_per_video * sample_every):
            ret, frame = cap.read()
            if not ret:
                break
            if fid % sample_every == 0:
                pred_mask, n_boxes, mode = run_yolo_segment(
                    frame, YOLO_MODEL, SEG_MODEL, fallback=True
                )
                pred_bin = (pred_mask > 127)
                total_frames += 1
                if is_negative:
                    if pred_bin.any():
                        fp_frames += 1
                else:
                    # No GT mask available for raw video → record detection rate
                    if pred_bin.any():
                        dices.append(1.0)  # placeholder; no GT available
                    else:
                        dices.append(0.0)
            fid += 1
        cap.release()
        n_videos_processed += 1

    if is_negative:
        fpr = round(fp_frames / max(total_frames, 1), 4)
        return {
            "name": name, "type": "video_negative",
            "status": "done",
            "n_videos": n_videos_processed,
            "n_frames_sampled": total_frames,
            "false_positive_frames": fp_frames,
            "false_positive_rate": fpr,
            "specificity": round(1.0 - fpr, 4),
            "is_negative": True,
        }

    return {
        "name": name, "type": "video_positive",
        "status": "done",
        "n_videos": n_videos_processed,
        "n_frames_sampled": total_frames,
        "detection_rate": round(np.mean(dices), 4) if dices else 0.0,
        "note": "No GT masks found for raw video — detection rate reported instead of dice.",
        "is_negative": False,
    }

print("Video evaluator ready ✓")


Video evaluator ready ✓


In [8]:
# ─────────────────────────────────────────────────────────────
# Run ALL evaluations
# ─────────────────────────────────────────────────────────────

ALL_RESULTS = []

def run_image_ds(name, ds_key, img_sub, mask_sub):
    root = RESOLVED.get(ds_key)
    if not root:
        print(f"  SKIP {name}: dataset not attached")
        return
    img_dir  = os.path.join(root, img_sub)
    mask_dir = os.path.join(root, mask_sub)
    print(f"  Evaluating {name} ...")
    r = evaluate_image_dataset(name, img_dir, mask_dir)
    ALL_RESULTS.append(r)
    if r.get("status") == "done":
        print(f"    dice={r['dice_mean']:.4f}±{r['dice_std']:.4f}  iou={r['iou_mean']:.4f}  n={r['n_evaluated']}")
    else:
        print(f"    SKIP: {r.get('reason','')}")

def run_video_ds(name, ds_key, is_neg=False):
    root = RESOLVED.get(ds_key)
    if not root:
        print(f"  SKIP {name}: dataset not attached")
        return
    print(f"  Evaluating {name} ({'negative control' if is_neg else 'positive'}) ...")
    r = evaluate_video_dataset(name, root, is_negative=is_neg)
    ALL_RESULTS.append(r)
    if r.get("status") == "done":
        if is_neg:
            print(f"    FPR={r['false_positive_rate']:.4f}  specificity={r['specificity']:.4f}")
        else:
            d = r.get('dice_mean')
            if d is not None:
                print(f"    dice={d:.4f}±{r.get('dice_std',0):.4f}  iou={r.get('iou_mean',0):.4f}")
            else:
                print(f"    detection_rate={r.get('detection_rate',0):.4f}  (no GT masks)")
    else:
        print(f"    SKIP: {r.get('reason','')}")

print("=" * 60)
print("IMAGE DATASETS")
print("=" * 60)

# CVC-300
run_image_ds("CVC-300",
    "endoscene-cvc300-polyp-raw-dataset",
    "CVC-300/Original", "CVC-300/Ground Truth")

# CVC-ClinicDB, Kvasir-SEG, ETIS-LARIB
for sub in DATASET_MAP["chakramodel-evaluation-datasets"]["sub_datasets"]:
    run_image_ds(sub["name"], "chakramodel-evaluation-datasets",
                 sub["img_sub"], sub["mask_sub"])

# HyperKvasir
run_image_ds("HyperKvasir-Seg",
    "hyperkvasir-dataset-first-half-and-and-ld-dataset",
    "hyper-kvasir-segmented-images-part 3/segmented-images/images",
    "hyper-kvasir-segmented-images-part 3/segmented-images/masks")

# PolypDB
for sub in DATASET_MAP["polypdb-polyp-raw"]["sub_datasets"]:
    run_image_ds(sub["name"], "polypdb-polyp-raw", sub["img_sub"], sub["mask_sub"])

print("\n" + "=" * 60)
print("VIDEO DATASETS")
print("=" * 60)

run_video_ds("PolypGen2021-Video",    "polypgen20021-video",             is_neg=False)
run_video_ds("LDPolyp-NoPolyp",      "ldpolypvideowithoutpolyps",       is_neg=True)
run_video_ds("LDPolyp-PolypOnly",    "ldpolypvideopolyponly",            is_neg=False)
run_video_ds("HyperKvasir-Video-P2A","hyperkvasir-labeled-videos-part2-001", is_neg=False)
run_video_ds("HyperKvasir-Video-P2B","hperkvasir-labeled-videos-part2-002",  is_neg=False)
run_video_ds("CVC-Video",            "cvc-sample-video",                is_neg=False)

print("\n✓ All evaluations complete.")


IMAGE DATASETS
  Evaluating CVC-300 ...
    SKIP: img_dir not found: /kaggle/input/datasets/CVC-300/Original
  Evaluating CVC-ClinicDB ...
    SKIP: img_dir not found: /kaggle/input/datasets/cvc-clinicdb/images
  Evaluating Kvasir-SEG ...
    SKIP: img_dir not found: /kaggle/input/datasets/kvasir-seg/images
  Evaluating ETIS-LARIB ...
    SKIP: img_dir not found: /kaggle/input/datasets/etis-larib/images
  Evaluating HyperKvasir-Seg ...
    SKIP: img_dir not found: /kaggle/input/datasets/hyper-kvasir-segmented-images-part 3/segmented-images/images
  SKIP PolypDB/Simula/NBI: dataset not attached
  SKIP PolypDB/Simula/WLI: dataset not attached
  SKIP PolypDB/BKAI/WLI: dataset not attached
  SKIP PolypDB/BKAI/NBI: dataset not attached
  SKIP PolypDB/BKAI/BLI: dataset not attached
  SKIP PolypDB/BKAI/FICE: dataset not attached
  SKIP PolypDB/BKAI/LCI: dataset not attached
  SKIP PolypDB/Karolinska: dataset not attached
  SKIP PolypDB/FICE: dataset not attached
  SKIP PolypDB/BLI: dataset no

In [9]:
# ─────────────────────────────────────────────────────────────
# Comprehensive comparison table
# ─────────────────────────────────────────────────────────────

import pandas as pd

rows = []
for r in ALL_RESULTS:
    if r.get("status") != "done":
        continue

    is_neg  = r.get("is_negative", False)
    ds_type = r.get("type", "unknown")

    if is_neg:
        rows.append({
            "Dataset":      r["name"],
            "Type":         "video-negative",
            "N Samples":    r.get("n_frames_sampled", "-"),
            "Dice Mean":    "N/A (negative ctrl)",
            "Dice Std":     "-",
            "IoU Mean":     "-",
            "Specificity":  f"{r['specificity']:.4f}",
            "FP Rate":      f"{r['false_positive_rate']:.4f}",
            "YOLO Detect%": "-",
            "Verdict":      "✓ PASS" if r['specificity'] >= 0.90 else "✗ FAIL",
        })
    elif "dice_mean" in r:
        yolo_pct = "-"
        if "yolo_detections" in r and "n_evaluated" in r and r["n_evaluated"] > 0:
            yolo_pct = f"{100*r['yolo_detections']/r['n_evaluated']:.0f}%"
        verdict = "✓ PASS" if r["dice_mean"] >= 0.60 else ("~ WEAK" if r["dice_mean"] >= 0.30 else "✗ FAIL")
        rows.append({
            "Dataset":      r["name"],
            "Type":         ds_type,
            "N Samples":    r.get("n_evaluated", r.get("n_frames_total", "-")),
            "Dice Mean":    f"{r['dice_mean']:.4f}",
            "Dice Std":     f"{r['dice_std']:.4f}",
            "IoU Mean":     f"{r['iou_mean']:.4f}",
            "Specificity":  "-",
            "FP Rate":      "-",
            "YOLO Detect%": yolo_pct,
            "Verdict":      verdict,
        })
    else:
        rows.append({
            "Dataset":      r["name"],
            "Type":         ds_type,
            "N Samples":    r.get("n_frames_sampled", "-"),
            "Dice Mean":    "-",
            "Dice Std":     "-",
            "IoU Mean":     "-",
            "Specificity":  "-",
            "FP Rate":      "-",
            "YOLO Detect%": f"{100*r.get('detection_rate',0):.0f}%",
            "Verdict":      "~ INFO (no GT masks)",
        })

df = pd.DataFrame(rows)
print("\n" + "=" * 90)
print("CHAKRAMODEL v8 — FULL EVALUATION COMPARISON TABLE")
print("=" * 90)
print(df.to_string(index=False))
print("=" * 90)

# Summary stats
image_rows = [r for r in ALL_RESULTS if r.get("status")=="done" and "dice_mean" in r and r.get("type")=="image"]
if image_rows:
    all_dice = [r["dice_mean"] for r in image_rows]
    print(f"\nImage datasets summary:")
    print(f"  Overall mean Dice : {np.mean(all_dice):.4f}")
    print(f"  Best dataset      : {max(image_rows, key=lambda x: x['dice_mean'])['name']} ({max(all_dice):.4f})")
    print(f"  Worst dataset     : {min(image_rows, key=lambda x: x['dice_mean'])['name']} ({min(all_dice):.4f})")

df.to_csv(f"{WORKING_DIR}/comparison_table_v8.csv", index=False)
print(f"\nSaved: {WORKING_DIR}/comparison_table_v8.csv")



CHAKRAMODEL v8 — FULL EVALUATION COMPARISON TABLE
Empty DataFrame
Columns: []
Index: []

Saved: /kaggle/working/comparison_table_v8.csv


In [10]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

image_rows = [r for r in ALL_RESULTS
              if r.get("status") == "done" and "dice_mean" in r]

if image_rows:
    names = [r["name"].split("/")[-1] for r in image_rows]
    means = [r["dice_mean"] for r in image_rows]
    stds  = [r["dice_std"]  for r in image_rows]
    colors = ["#2ecc71" if m >= 0.60 else "#e67e22" if m >= 0.30 else "#e74c3c" for m in means]

    fig, ax = plt.subplots(figsize=(max(12, len(names)*0.7), 6))
    bars = ax.bar(names, means, yerr=stds, capsize=4,
                  color=colors, alpha=0.85, edgecolor="white", linewidth=0.5)
    ax.axhline(0.60, color="#2ecc71", linestyle="--", linewidth=1.2, label="Good threshold (0.60)")
    ax.axhline(0.30, color="#e74c3c", linestyle="--", linewidth=1.2, label="Poor threshold (0.30)")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Dice Score")
    ax.set_title("ChakraModel v8 — Dice Score per Dataset", fontsize=13, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    p1 = mpatches.Patch(color="#2ecc71", alpha=0.85, label="≥0.60 PASS")
    p2 = mpatches.Patch(color="#e67e22", alpha=0.85, label="0.30-0.60 WEAK")
    p3 = mpatches.Patch(color="#e74c3c", alpha=0.85, label="<0.30 FAIL")
    ax.legend(handles=[p1, p2, p3], loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{WORKING_DIR}/dice_comparison_v8.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Plot saved.")


In [11]:
import json as _json

report = {
    "version": "v8",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "n_datasets_evaluated": len([r for r in ALL_RESULTS if r.get("status")=="done"]),
    "results": ALL_RESULTS,
}

out = f"{WORKING_DIR}/combined_report_v8.json"
with open(out, "w") as f:
    _json.dump(report, f, indent=2, default=str)
print(f"Full report saved: {out}")

# Zip artifacts
os.chdir(WORKING_DIR)
os.system("zip -r chakra_v8_artifacts.zip combined_report_v8.json comparison_table_v8.csv dice_comparison_v8.png > /dev/null")
print("Done! Download chakra_v8_artifacts.zip from the output panel.")


Full report saved: /kaggle/working/combined_report_v8.json
Done! Download chakra_v8_artifacts.zip from the output panel.
